# Homología Persistente

En esta libreta, calcularemos la homología persistente de algunas filtraciones.

Empezaremos instalando algunas librerías que nos haran falta.

In [ ]:
%%capture
pip install gudhi networkx trimesh

Por otro lado, vamos a instalar el módulo PHAT (https://www.sciencedirect.com/science/article/pii/S0747717116300098) que nos permitirá trabajar con las matrices sobre $\mathbb{Z}_2$. Para esto utilizaremos una versión algo modificada.

In [ ]:
# %%capture
# !pip install setuptools pybind11
# !pip install --no-build-isolation git+https://bitbucket.org/atorras1618/phat.git

## Part 1: Filtraciones de estrella inferior

En diversas situaciones, uno dispone de una triangulación sin tener una filtración natural impuesta por los datos. O quizás uno prefiere calcular una filtración dada por la disposición espacial de la triangulación. En este contexto, la filtración de estrella inferior se presenta como una filtración muy natural. En la siguiente figura, podemos ver la triangulación de una figura conocida, el "conejo de Stanford", con la filtración por subniveles de estrella inferior determinada por la dirección `(1,1,0)`. A continuación, veremos como construir esta filtración y calcularemos su homología persistente.

![Bunny](images/bunny_sublevel.png)

Empezaremos descargando la triangulación del conejo de Stanford, si aún no lo hemos hecho. Durante esta práctica, utilizaremos una triangulación que utiliza menos triángulos que el objeto original, esta reducción se encuentra en la librería `libigl` y lo obtendremos mediante su URL.

In [ ]:
import urllib.request
import os
import trimesh

# empezamos descargando la triangulación de una forma conocida como el "conejo de Stanford"
# Usamos una reducción de la triangulación presente in el siguiente URL:
url = "https://raw.githubusercontent.com/libigl/libigl-tutorial-data/master/bunny.off"
filename = "class_bunny.off"
if not os.path.exists(filename):
    urllib.request.urlretrieve(url, filename)
# obtenemos los vértices y triángulos de la triangulación
mesh = trimesh.load(filename)
vertices = mesh.vertices
triangles = mesh.faces 

In [ ]:
import gudhi
st_bunny = gudhi.SimplexTree()

In [ ]:
import numpy as np
st_bunny.insert_batch(triangles.transpose(), np.zeros(triangles.shape[0]))

In [ ]:
import matplotlib.pyplot as plt
from funciones_auxiliares import plot_simplex_tree_3D

plot_simplex_tree_3D(st_bunny, vertices, alpha_faces=0.7)
# ajustamos la cámara mediante el siguiente comando
plt.gca().view_init(elev=100, azim=-90)
plt.tight_layout()
plt.show()

Como en la libreta anterior, podemos inspeccionar la información del complejo simplicial `st_bunny`, incluyendo los números de betti. Para capturar $\beta_2$, aumentamos la dimensión del complejo simplicial a $3$.

In [ ]:
st_bunny.set_dimension(3)
print("Información sobre el complejo simplicial bunny:")
print(f"Dimensión: {st_bunny.dimension()}")
print(f"Número de vértices: {st_bunny.num_vertices()}")
print(f"Número de símplices: {st_bunny.num_simplices()}")
st_bunny.compute_persistence()
print(f"Números de Betti: {st_bunny.betti_numbers()}")

In [ ]:
from funciones_auxiliares import height_filtration_from_mesh

st_bunny = height_filtration_from_mesh(mesh, direction=[1,1,0])

In [ ]:
plot_simplex_tree_3D(st_bunny, vertices, alpha_faces=0.7)
# ajustamos la cámara mediante el siguiente comando
plt.gca().view_init(elev=100, azim=-90)
plt.tight_layout()
plt.show()

In [ ]:
from funciones_auxiliares import diccionario_simplices
import os


num_slices = 5
first_sublevel = 0.01
shift = 0.045
fig = plt.figure(figsize=(num_slices*5, 6))
for idx in range(num_slices):
    ax = fig.add_subplot(1, num_slices, idx + 1, projection='3d')
    st_aux = st_bunny.copy()
    st_aux.prune_above_filtration(first_sublevel+idx*shift)
    plot_simplex_tree_3D(st_aux, vertices, alpha_faces=0.7, ax=ax)
    ax.view_init(elev=100, azim=-90)
    st_aux.set_dimension(3)
    st_aux.compute_persistence()
    ax.set_title(f"Betti: {st_aux.betti_numbers()}", fontsize=20)
plt.tight_layout()
plt.savefig(os.path.join("images", "bunny_sublevel.png"), dpi=50)

Como podemos comprobar, los números de Betti cambian con el valor máximo de filtración que tomamos para el subcomplejo de nivel. Podemos calcular directamente todos los números de Betti mediante la homología persistente.

In [ ]:
st_bunny.set_dimension(3)
st_bunny.compute_persistence()
st_bunny.persistent_betti_numbers(0.09, 0.12)

In [ ]:
import gudhi
diag = st_bunny.persistence()
gudhi.plot_persistence_barcode(diag)
plt.show()

In [ ]:
st_bunny.lower_star_persistence_generators()

In [ ]:
import networkx as nx
import numpy as np
import sympy as sp

import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
from mpl_toolkits.mplot3d.art3d import Poly3DCollection, Line3DCollection
import matplotlib.cm as cm
import matplotlib.colors as mcolors

import gudhi

def plot_simplex_tree_3D(st, points, alpha_faces=0.5, figsize=(5,5), use_filtration=True, ax=None, plot_lower_star_generators=False):
    """ Función para visualizar un complejo simplicial:
    st: complejo simplicial, estructura `simplex_tree`de Gudhi
    points: puntos en formato numpy.array (numero de puntos, 3) 
    """
    # Vamos a extraer y agrupar las aristas y los triángulos a partir de st
    vertices = []
    edges = []
    triangles = []
    triangle_filtrations = []
    for simplex, filtration in st.get_skeleton(2): # We only need up to 2D faces for 3D visualization
        dim = len(simplex) - 1
        if dim == 0:
            vertices.append(simplex[0])
        elif dim == 1:
            edges.append(simplex)
        elif dim == 2:
            triangles.append(simplex)
            triangle_filtrations.append(filtration)
            
    # Initialize the figure if axis not given
    if ax is None:
        fig = plt.figure(figsize=figsize)
        ax = fig.add_subplot(111, projection='3d')
    # Plot all Vertices in one command
    vtx_coords = points[vertices]
    ax.scatter(vtx_coords[:, 0], vtx_coords[:, 1], vtx_coords[:, 2], 
                color='black', s=10, zorder=3)
    
    # Plot all Edges in one command using Line3DCollection
    if edges:
        edge_coords = points[np.array(edges)] # Shape: (num_edges, 2, 3)
        edge_collection = Line3DCollection(edge_coords, colors='black', linewidths=0.2, alpha=0.4, zorder=2)
        ax.add_collection3d(edge_collection)
    
    # Plot all Triangles (Faces) in one command using Poly3DCollection
    if triangles:
        if use_filtration:
            tri_coords = points[np.array(triangles)] 
            filtrations_array = np.array(triangle_filtrations)
            norm = mcolors.Normalize(vmin=filtrations_array.min(), vmax=filtrations_array.max())
            face_colors = cm.plasma_r(norm(filtrations_array))
        else:
            tri_coords = points[np.array(triangles)] 
            # tomamos el centroide de cada triangulo
            centroids = np.mean(tri_coords, axis=1) 
            # tomamos las coordenadas z del centroide de cada triangulo
            z_centers = centroids[:, 2] 
            # normalizamos los centroides de 0 a 1 para el colormap
            norm = mcolors.Normalize(vmin=z_centers.min(), vmax=z_centers.max())
            # calculamos los colores de cada triangulo segun z_centers
            face_colors = cm.plasma_r(norm(z_centers))
        # 6. Pass the color array to facecolors
        tri_collection = Poly3DCollection(tri_coords, facecolors=face_colors, edgecolors='none', alpha=alpha_faces, zorder=1)
        ax.add_collection3d(tri_collection)

    # Plot lower star generators 
    if plot_lower_star_generators:
        # compute_persistence MUST be called before extracting generators in GUDHI
        st.compute_persistence()
        
        
        
        # lower_star_persistence_generators returns (regular_pairs, essential_features)
        regular_pairs, essential_features = st.lower_star_persistence_generators()
        # Loop through dimensions using the returned regular pairs
        for dim, pairs_in_dim in enumerate(regular_pairs):
            if len(pairs_in_dim) == 0:
                continue # Skip if there are no generators in this dimension
                
            # Extract birth and death vertex indices
            birth_vertices = pairs_in_dim[:, 0]
            death_vertices = pairs_in_dim[:, 1]

            # Grab the specific color for this dimension using Set1
            color = cm.Set1(dim)
            
            # Map indices to their respective 3D coordinates
            birth_coords = points[birth_vertices]
            death_coords = points[death_vertices]
            
            # Range over pairs
            for i, (b_coord, d_coord) in enumerate(zip(birth_coords, death_coords)):
                # Plot a line joining persistence pairs
                ax.plot([b_coord[0], d_coord[0]], 
                        [b_coord[1], d_coord[1]], 
                        [b_coord[2], d_coord[2]], 
                        color=color, linewidth=3, zorder=1000)

                # Legends 
                label_birth = f"Birth Dim {dim}" if i == 0 else "_nolegend_"
                label_death = f'Death Dim {dim}' if i == 0 else "_nolegend_"
                # Birth point
                ax.plot([b_coord[0]], [b_coord[1]], [b_coord[2]], 
                        marker='o', markersize=8, color=color, markeredgecolor='white', 
                        linestyle='None', zorder=1001, label=label_birth)
                
                # Death point
                ax.plot([d_coord[0]], [d_coord[1]], [d_coord[2]], 
                        marker='X', markersize=8, color=color, markeredgecolor='white', 
                        linestyle='None', zorder=1001, label=label_death)

        # Loop through dimensions using the essential features
        for dim, features_in_dim in enumerate(essential_features):
            if len(features_in_dim) == 0:
                continue # Skip if there are no generators in this dimension

            # Grab the specific color for this dimension using Set1
            color = cm.Set1(dim)
        
            # Map indices to their respective 3D coordinates
            features_coords = points[features_in_dim]
            
            # Range over pairs
            for i, coord in enumerate(features_coords):
                label_feature = f"Feature Dim {dim}" if i == 0 else "_nolegend_"
                # feature coord (plot)
                ax.plot([b_coord[0]], [b_coord[1]], [b_coord[2]], 
                        marker='s', markersize=8, color=color, markeredgecolor='white', 
                        linestyle='None', zorder=1002, label=label_feature)
                
        # Optional: You can uncomment this to show a legend for the dimensions
        # ax.legend(loc="upper right")
    # ---------------------- 
    x_min, x_max = points[:, 0].min(), points[:, 0].max()
    y_min, y_max = points[:, 1].min(), points[:, 1].max()
    z_min, z_max = points[:, 2].min(), points[:, 2].max()
    # 2. Force the 3D axis limits to frame the entire point cloud
    ax.set_xlim([x_min, x_max])
    ax.set_ylim([y_min, y_max])
    ax.set_zlim([z_min, z_max])
    # Set aspect proportional
    ax.set_box_aspect((x_max - x_min, y_max - y_min, z_max - z_min))
    

In [ ]:

plot_simplex_tree_3D(st_bunny, vertices, plot_lower_star_generators=True)
# ajustamos la cámara mediante el siguiente comando
plt.gca().view_init(elev=100, azim=-90)
plt.tight_layout()
plt.legend()
plt.show()

In [ ]:
st_bunny.lower_star_persistence_generators()

In [ ]:
ax = gudhi.plot_persistence_diagram(diag)
# We can modify the title, aspect, etc.
ax.set_title("Persistence diagram of a torus")
ax.set_aspect("equal")  # forces to be square shaped
plt.show()

### Indices de vertices

Vamos a calcular los vértices de la filtración cuyo valor es distinto de cero.

### Bottleneck Stability

Compute bottleneck distance with respect to another direction.

In [ ]:
st_bunny_2 = height_filtration_from_mesh(mesh, direction=[1.1,0.9,0])
st_bunny_2.set_dimension(3)
st_bunny_2.compute_persistence()
diag_dim_1 = st_bunny.persistence_intervals_in_dimension(1)
diag_2_dim_1 = st_bunny_2.persistence_intervals_in_dimension(1)
gudhi.bottleneck_distance(diag_dim_1, diag_2_dim_1)